# Demo del pipeline ETL completo

Recorrido paso a paso: **Extract → Transform → Validate → Load**.
Reutiliza los módulos de producción en `etl/`.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "etl").exists():
    ROOT = ROOT.parent.parent
sys.path.insert(0, str(ROOT))

from etl.config import get_settings
from etl.extract.orchestrator import run_extraction
from etl.transform.pollution_transformer import compute_correlations, transform_pollution
from etl.transform.weather_transformer import transform_weather
from etl.validation.validator import validate_datasets

settings = get_settings()
settings.processed_data_dir.mkdir(parents=True, exist_ok=True)
print(f"Período: {settings.date_start} → {settings.date_end}")

## Etapa 1: Extract

In [ ]:
pollution_dfs, weather_raw, snapshot_dir = run_extraction()
print(f"Snapshot guardado en: {snapshot_dir}")
print(f"Contaminantes: {list(pollution_dfs.keys())}")
print(f"Registros clima: {len(weather_raw)}")

## Etapa 2: Transform

In [ ]:
pollution_df, daily_df, monthly_df = transform_pollution(pollution_dfs)
weather_df = transform_weather(weather_raw)
correlation_df = compute_correlations(pollution_df, weather_df)

pollution_df.to_parquet(settings.processed_data_dir / "pollution.parquet", index=False)
weather_df.to_parquet(settings.processed_data_dir / "weather.parquet", index=False)
daily_df.to_parquet(settings.processed_data_dir / "daily_metrics.parquet", index=False)
monthly_df.to_parquet(settings.processed_data_dir / "monthly_metrics.parquet", index=False)
correlation_df.to_parquet(settings.processed_data_dir / "correlations.parquet", index=False)

print(f"Mediciones contaminación: {len(pollution_df)}")
print(f"Métricas diarias: {len(daily_df)}")
print(f"Correlaciones calculadas: {len(correlation_df)}")
display(pollution_df.head())

## Etapa 3: Validate

In [ ]:
report = validate_datasets(pollution_df, weather_df)
report_path = settings.processed_data_dir / "validation_reports"
report_path.mkdir(parents=True, exist_ok=True)
out_file = report_path / "validation_notebook.json"
with open(out_file, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, default=str)

print(f"Validación global: {'OK' if report['overall_success'] else 'WARNING'}")
print(json.dumps(report, indent=2, default=str))

## Etapa 4: Load (requiere PostgreSQL activo)

Ejecutar solo si la base de datos está disponible (`docker compose up postgres` o stack completo).

In [ ]:
import os

LOAD_TO_DB = os.getenv("NOTEBOOK_LOAD_DB", "false").lower() == "true"

if LOAD_TO_DB:
    from scripts.migrate import migrate
    from etl.load.db_loader import load_all_data

    migrate()
    total = load_all_data(
        pollution_df, weather_df, daily_df, monthly_df, correlation_df, run_id="notebook"
    )
    print(f"Registros cargados en PostgreSQL: {total}")
else:
    print("Carga omitida. Para cargar: NOTEBOOK_LOAD_DB=true y PostgreSQL activo.")
    print("Alternativa: docker compose run --rm etl")